# BEE 4750 Homework 5: Mixed Integer and Stochastic Programming

**Name**: Sarah Vafiadis, Nathan Rhoads

**ID**: sv439, nar84

> **Due Date**
>
> Thursday, 12/04/24, 9:00pm

## Overview

### Instructions

-   In Problem 1, you will use mixed integer programming to solve a
    waste load allocation problem.
-   In Problem 2, you will formulate a stochastic optimization problem.

### Load Environment

The following code loads the environment and makes sure all needed
packages are installed. This should be at the start of most Julia
scripts.

In [4]:
import Pkg
Pkg.activate(@__DIR__)
Pkg.instantiate()

  Activating project at `c:\Users\sarah\OneDrive\Documents\GitHub\hw5-sarah_nathan_hw5`


In [5]:
using JuMP
using HiGHS
using DataFrames
using GraphRecipes
using Plots
using Measures
using MarkdownTables

## Problems (Total: 30 Points)

### Problem 1 (24 points)

Three cities are developing a coordinated municipal solid waste (MSW)
disposal plan. Three disposal alternatives are being considered: a
landfill (LF), a materials recycling facility (MRF), and a
waste-to-energy facility (WTE). The capacities of these facilities and
the fees for operation and disposal are provided below.

-   **LF**: Capacity 200 Mg, fixed cost \$2000/day, tipping cost
    \$50/Mg;
-   **MRF**: Capacity 350 Mg, fixed cost \$1500/day, tipping cost
    \$7/Mg, recycling cost \$40/Mg recycled;
-   **WTE**: Capacity 210 Mg, fixed cost \$2500/day, tipping cost
    \$60/Mg;

The MRF recycling rate is 40%, and the ash fraction of non-recycled
waste is 16% and of recycled waste is 14%. Transportation costs are
\$1.5/Mg-km, and the relative distances between the cities and
facilities are provided in the table below.

| **City/Facility** | **Landfill (km)** | **MRF (km)** | **WTE (km)** |
|:-----------------:|:-----------------:|:------------:|:------------:|
|         1         |         5         |      30      |      15      |
|         2         |        15         |      25      |      10      |
|         3         |        13         |      45      |      20      |
|        LF         |        \-         |      32      |      18      |
|        MRF        |        32         |      \-      |      15      |
|        WTE        |        18         |      15      |      \-      |

The fixed costs associated with the disposal options are incurred only
if the particular disposal option is implemented. The three cities
produce 100, 90, and 120 Mg/day of solid waste, respectively, with the
composition provided in the table below.

| **Component** | **% of total mass** | **Combustion ash** (%) | **MRF Recycling rate** (%) |
|:---------------------:|:--------------:|:---------------:|:---------------:|
| Food Wastes | 15 | 8 | 0 |
| Paper & Cardboard | 40 | 7 | 55 |
| Plastics | 5 | 5 | 15 |
| Textiles | 3 | 10 | 10 |
| Rubber, Leather | 2 | 15 | 0 |
| Wood | 5 | 2 | 30 |
| Yard Wastes | 18 | 2 | 40 |
| Glass | 4 | 100 | 60 |
| Ferrous | 2 | 100 | 75 |
| Aluminum | 2 | 100 | 80 |
| Other Metal | 1 | 100 | 50 |
| Miscellaneous | 3 | 70 | 0 |

The information in the above table will help you determine the overall
recycling and ash fractions. Note that the recycling residuals, which
may be sent to either landfill or the WTE, have different ash content
than the ash content of the original MSW. You will need to determine
these fractions to construct your mass balance constraints.

**Reminder**: Use `round(x; digits=n)` to report values to the
appropriate precision!

#### Problem 1.1

Based on the information above, calculate the overall recycling and ash
fractions for the waste produced by each city.

In [6]:
# Component table (mass share, ash %, MRF recycle %)
comp = [
    (0.15, 0.08, 0.00),     # Food waste
    (0.40, 0.07, 0.55),     # Paper and cardboard
    (0.05, 0.05, 0.15),     # Plastics
    (0.03, 0.10, 0.10),     # Textiles
    (0.02, 0.15, 0.00),     # Rubber, leather
    (0.05, 0.02, 0.30),     # Wood
    (0.18, 0.02, 0.40),     # Yard wastes
    (0.04, 1.00, 0.60),     # Glass
    (0.02, 1.00, 0.75),     # Ferrous
    (0.02, 1.00, 0.80),     # Aluminum
    (0.01, 1.00, 0.50),     # Other metals
    (0.03, 0.70, 0.00)      # Misc
]

# Calculate fractions
recycle = sum(x[1] * x[3] for x in values(comp))
ash = sum(x[1] * x[2] for x in values(comp))

println("Recycling fraction: ", round(recycle, digits=3))
println("Ash fraction: ", round(ash, digits=3))

Recycling fraction: 0.378
Ash fraction: 0.164


#### Problem 1.2

What are the decision variables for your optimization problem? Provide
notation and variable meaning.

##### Decision variables

$W_{ij}$    Waste transported from source i to disposal j (mg/day)

$R_{kj}$    Residual waste transported from disposal k to disposal j (mg/day)

$Y_j$       Operational status of disposal j (binary); 0 if facility not operated and 1 if facility operated

##### Additional variables

i = cities 1,2,3

j = LF, MRF, WTE

$d_{ij}$    Distance between city i and facility j

#### Problem 1.3

Formulate the objective function. Make sure to include any needed
derivations or justifications for your equation(s).

Objective: minimize Total Cost

= Fixed Costs + Transportation Costs + Disposal Costs (tipping/recycling)

Fixed cost = $2000Y_{LF} + 1500Y_{MRF} + 2500Y_{WTE}$

Fixed cost for each facility multiplied by operational status

Transportation cost = $ 1.5 \sum_{i=1}^{3} \sum_{j=LF,MRF,WTE}d_{ij} W_{ij}$

Multiply fixed transportation cost by sum of distance between cities and facilities and waste sent from each city to each facility

Tipping cost = $ 50(\sum_{i} W_{i,LF} + R_{MRF,LF} + R_{WTE,LF})+ 40( \sum_{i} W_{i,MRF}) + 60(\sum_{i} W_{i,WTE} + R_{MRF,WTE})$

Multiply tipping cost for each facility by waste input into facility and residual waste delivered by other facilities

Recycling cost = $40 (0.4W_{i,MRF})$

40% recycling rate and $40/mg

#### Objective function

$minimize \hspace{0.5cm} 2000Y_{LF} + 1500Y_{MRF} + 2500Y_{WTE} + 1.5 \sum_{i=1}^{3} \sum_{j=LF,MRF,WTE}d_{ij} W_{ij} + 50(\sum_{i} W_{i,LF} + R_{MRF,LF} + R_{WTE,LF})+ 40( \sum_{i} W_{i,MRF}) + 60(\sum_{i} W_{i,WTE} + R_{MRF,WTE})$

#### Problem 1.4

Derive all relevant constraints. Make sure to include any needed justifications or derivations.

non-negativity and binary constraint: $W_{i,j}, R_{k,j}, >= 0$, $Y_{j} $= 0 or 1 

city balance of waste: $\sum_{W_{i,j}}$ = City waste

MRF mass balance: $R_{MRF,LF} + R_{MRF,WTE} $= (1-.4) $\sum_{W_{i,MRF}} $

WTE ash production: $R_{WTE,LF} = .16\sum{W_{i,WTE}} + .16R_{MRF, WTE} $

Capacities:

$R_{MRF,LF} + R_{WTE,LF} + \sum_{W_{i, LF}} <= 200Y_{LF} $

$\sum_{W_{i,MRF}} <= 350Y_{MRF} $

$\sum_{W_{i,WTE}} + R_{MRF,WTE} <= 210Y_{WTE} $





#### Problem 1.5

Find the optimal solution (using `JuMP` to solve the problem). Report
the optimal objective value.

In [26]:
# Data

facilities = [:LF, :MRF, :WTE]

# City waste generation (mg/d)
cities = 1:3
waste = Dict(1 => 100.0, 2 => 90.0, 3 => 120.0)
resid_source = [:MRF, :WTE]

# Distances from facilities to cities (km)
dist_city = Dict(
    :LF  => Dict(1 => 5.0,  2 => 15.0, 3 => 13.0),
    :MRF => Dict(1 => 30.0, 2 => 25.0, 3 => 45.0),
    :WTE => Dict(1 => 15.0, 2 => 10.0, 3 => 20.0)
)

# Distances between facilities (km)
dist_fac = Dict(
    (:MRF, :LF)  => 32.0,
    (:MRF, :WTE) => 15.0,
    (:WTE, :LF)  => 18.0
)

# Capacities (Mg/day)
cap = Dict(:LF => 200.0, :MRF => 350.0, :WTE => 210.0)

# Fixed costs ($/day)
F = Dict(:LF => 2000.0, :MRF => 1500.0, :WTE => 2500.0)

# Tipping costs ($/mg)
tipping = Dict(:LF => 50.0, :MRF => 7.0, :WTE => 60.0)

transport = 1.5     # Transportation costs ($/mg-km)
recycle = 40.0      # Recycling costs ($/mg recycled)
MRF_recycle = 0.4   # MRF recycling rate

# Model

total_cost = Model(HiGHS.Optimizer)

@variable(total_cost, W[i in cities, j in facilities] >= 0)     # W_ij
@variable(total_cost, R[k in resid_source, j in facilities] >= 0)    # R_kj
@variable(total_cost, Y[j in facilities], Bin)                  # Y_j

@constraint(total_cost, [i in cities],                      # All waste is delivered somewhere
    sum(W[i, j] for j in facilities) == waste[i]
)

@objective(total_cost, Min,
    # Fixed cost
    2000 * Y[:LF] + 1500 * Y[:MRF] + 2500 * Y[:WTE] +

    # Transport cost
    1.5 * sum(dist_city[j][i] * W[i, j] for i in cities, j in facilities) +

    # Tipping and recycling
    50 * (sum(W[i, :LF] for i in cities) + R[:MRF, :LF] + R[:WTE, :LF]) + 40 * sum(W[i, :MRF] for i in cities) +
    60 * (sum(W[i, :WTE] for i in cities) + R[:MRF, :WTE])
)

optimize!(total_cost)
println("\nTotal Cost: ", objective_value(total_cost))

Running HiGHS 1.12.0 (git hash: 755a8e027): Copyright (c) 2025 HiGHS under MIT licence terms
MIP has 3 rows; 18 cols; 9 nonzeros; 3 integer variables (3 binary)
Coefficient ranges:
  Matrix  [1e+00, 1e+00]
  Cost    [5e+01, 2e+03]
  Bound   [1e+00, 1e+00]
  RHS     [9e+01, 1e+02]
Presolving model
3 rows, 3 cols, 3 nonzeros  0s
0 rows, 0 cols, 0 nonzeros  0s
Presolve reductions: rows 0(-3); columns 0(-18); nonzeros 0(-9) - Reduced to empty
Presolve: Optimal

Src: B => Branching; C => Central rounding; F => Feasibility pump; H => Heuristic;
     I => Shifting; J => Feasibility jump; L => Sub-MIP; P => Empty MIP; R => Randomized rounding;
     S => Solve LP; T => Evaluate node; U => Unbounded; X => User solution; Y => HiGHS solution;
     Z => ZI Round; l => Trivial lower; p => Trivial point; u => Trivial upper; z => Trivial zero

        Nodes      |    B&B Tree     |            Objective Bounds              |  Dynamic Constraints |       Work      
Src  Proc. InQueue |  Leaves   Expl. |

#### Problem 1.6

Draw a diagram showing the flows of waste between the cities and the
facilities. Which facilities (if any) will not be used? Does this
solution make sense?

In [2]:
using JuMP, HiGHS

cities = 1:3
facilities = [:LF, :MRF, :WTE]
S = Dict(1=>100.0, 2=>90.0, 3=>120.0)
dist = Dict((1,:LF)=>5, (1,:MRF)=>30, (1,:WTE)=>15,
            (2,:LF)=>15,(2,:MRF)=>25,(2,:WTE)=>10,
            (3,:LF)=>13,(3,:MRF)=>45,(3,:WTE)=>20)
transport_cost = 1.5
tipping = Dict(:LF=>50.0, :MRF=>7.0, :WTE=>60.0)
fixed = Dict(:LF=>2000.0, :MRF=>1500.0, :WTE=>2500.0)
capacity = Dict(:LF=>200.0, :MRF=>350.0, :WTE=>210.0)
MRF_recycle = 0.40
recy_process_cost = 40.0
ash_non = 0.16
ash_res = 0.14  # choose interpretation

model = Model(HiGHS.Optimizer)

@variable(model, W[cities, facilities] >= 0)
@variable(model, R_MRF_LF >= 0)
@variable(model, R_MRF_WTE >= 0)
@variable(model, R_WTE_LF >= 0)
@variable(model, Y[facilities], Bin)

# city balances
for i in cities
    @constraint(model, sum(W[i,j] for j in facilities) == S[i])
end

# MRF residuals
@constraint(model, R_MRF_LF + R_MRF_WTE == (1-MRF_recycle)*sum(W[i,:MRF] for i in cities))

# WTE ash
@constraint(model, R_WTE_LF == ash_non*sum(W[i,:WTE] for i in cities) + ash_res*R_MRF_WTE)

# capacities
@constraint(model, sum(W[i,:LF] for i in cities) + R_MRF_LF + R_WTE_LF <= capacity[:LF]*Y[:LF])
@constraint(model, sum(W[i,:MRF] for i in cities) <= capacity[:MRF]*Y[:MRF])
@constraint(model, sum(W[i,:WTE] for i in cities) + R_MRF_WTE <= capacity[:WTE]*Y[:WTE])

# objective: fixed + transport + tipping + recycling processing
transport_term = sum(transport_cost*dist[(i,j)]*W[i,j] for i in cities for j in facilities)
tipping_term = sum(tipping[j]*W[i,j] for i in cities for j in facilities)
recy_term = recy_process_cost * MRF_recycle * sum(W[i,:MRF] for i in cities)
fixed_term = sum(fixed[j]*Y[j] for j in facilities)

@objective(model, Min, fixed_term + transport_term + tipping_term + recy_term)

optimize!(model)

# read flows:
for i in cities, j in facilities
    println("W[$i,$(j)] = ", value(W[i,j]))
end
println("R_MRF_LF = ", value(R_MRF_LF))
println("R_MRF_WTE = ", value(R_MRF_WTE))
println("R_WTE_LF = ", value(R_WTE_LF))
println("Y = ", Dict(j => value(Y[j]) for j in facilities))

Running HiGHS 1.12.0 (git hash: 755a8e027): Copyright (c) 2025 HiGHS under MIT licence terms
MIP has 8 rows; 15 cols; 34 nonzeros; 3 integer variables (3 binary)
Coefficient ranges:
  Matrix  [1e-01, 4e+02]
  Cost    [6e+01, 2e+03]
  Bound   [1e+00, 1e+00]
  RHS     [9e+01, 1e+02]
Presolving model
8 rows, 15 cols, 34 nonzeros  0s
7 rows, 13 cols, 31 nonzeros  0s
Presolve reductions: rows 7(-1); columns 13(-2); nonzeros 31(-3) 

Solving MIP model with:
   7 rows
   13 cols (2 binary, 0 integer, 0 implied int., 11 continuous, 0 domain fixed)
   31 nonzeros

Src: B => Branching; C => Central rounding; F => Feasibility pump; H => Heuristic;
     I => Shifting; J => Feasibility jump; L => Sub-MIP; P => Empty MIP; R => Randomized rounding;
     S => Solve LP; T => Evaluate node; U => Unbounded; X => User solution; Y => HiGHS solution;
     Z => ZI Round; l => Trivial lower; p => Trivial point; u => Trivial upper; z => Trivial zero

        Nodes      |    B&B Tree     |            Objective 

### Problem 2 (6 points)

Consider a two-period economic dispatch problem, based on the
multi-period example from Lecture 14 (on 10/29). The generator data,
including ramping constraints for each generator, is provided in
\`data/generators.csv.’ In period 1, the demand is
$d_1 = 1100 \text{MW}$. In period 2, the demand is projected to be
$d_2 = 1200 \text{MW}$, but there is a 25% probability that it is \$1500
. In the first period, the solar capacity factor is $0.9$ and the wind
capacity factor is $0.45$, but in the second period, there is some
uncertainty: the forecasted solar and wind capacity factors are $0.95$
and $0.4$, respectively, but there is a 30% probability that they are
$0.75$ and $0.5$. Your goal is to identify how to dispatch your
generators to minimize the cost of meeting demand.

#### Problem 2.1

Draw a scenario tree for this problem.

#### Problem 2.2

Formulate a stochastic linear program for this problem based on your
scenario tree from Problem 2.1 and the data in `data/generators.csv`.

In [25]:
G = ["Biomass","Hydro","Geo","NG_CCGT","NG_CT","Wind","Solar"] #G is the different generation types

Pmin = Dict("Biomass"=>0,"Hydro"=>0,"Geo"=>0,"NG_CCGT"=>100,"NG_CT"=>100,"Wind"=>0,"Solar"=>0) #minimum power output
Pmax = Dict("Biomass"=>100,"Hydro"=>500,"Geo"=>400,"NG_CCGT"=>250,"NG_CT"=>250,"Wind"=>300,"Solar"=>500) #maximum power output
Cost = Dict("Biomass"=>5,"Hydro"=>0,"Geo"=>0,"NG_CCGT"=>23,"NG_CT"=>38,"Wind"=>0,"Solar"=>0) #cost per MWh per tech
Ramp = Dict("Biomass"=>100,"Hydro"=>500,"Geo"=>400,"NG_CCGT"=>100,"NG_CT"=>200,"Wind"=>300,"Solar"=>500) #ramping limits

d1 = 1100.0 #demand in period 1

#different scenarios for period 2 from the tree, including (name, demand, solar_cf, wind_cf, probability)
scenarios = [
    ("S1", 1200.0, 0.95, 0.40, 0.525),
    ("S2", 1200.0, 0.75, 0.50, 0.225),
    ("S3", 1500.0, 0.95, 0.40, 0.175),
    ("S4", 1500.0, 0.75, 0.50, 0.075)
]

#renewable output (fixed)
solar1 = Pmax["Solar"] * 0.90 #500 * .9 = 450
wind1 = Pmax["Wind"] * 0.45 #300 * .45 = 135

m = Model(HiGHS.Optimizer) #write the optimization model

#p1(g) = generation in period 1 MWh
@variable(m, p1[g in G], lower_bound=Pmin[g], upper_bound=Pmax[g])

#fix renewables p1 so they are not decision variables
fix(p1["Solar"], solar1; force=true) #help from stackoverflow, still unsure what this does entirely
fix(p1["Wind"],  wind1; force=true)

#p2(g,s)
@variable(m, p2[g in G, s in 1:4], lower_bound=Pmin[g], upper_bound=Pmax[g])

for s in 1:length(scenarios)
    scenario = scenarios[s]

    solar_cf = scenario[3] #third
    wind_cf  = scenario[4] #fourth

    fix(p2["Solar", s], Pmax["Solar"] * solar_cf; force=true) 
    fix(p2["Wind",  s], Pmax["Wind"]  * wind_cf;  force=true)
end

#p1
@constraint(m, sum(p1[g] for g in G) == d1)

#p2 (one per scenario)
for s in 1:length(scenarios)
    d2 = scenarios[s][2] #second
    @constraint(m, sum(p2[g, s] for g in G) == d2)
end
#ramping constraints
for g in G
    if g != "Solar" && g != "Wind"
        RU = Ramp[g]
        for s in 1:length(scenarios)
            @constraint(m, p2[g,s] - p1[g] <= RU)
            @constraint(m, p1[g] - p2[g,s] <= RU)
        end
    end
end
#objective
@objective(m, Min,
    sum(Cost[g] * p1[g] for g in G) +
    sum(
        scenarios[s][5] * Cost[g] * p2[g,s]
        for s in 1:length(scenarios), g in G
    )
)
#optimize
optimize!(m)
#start printing the results 
println("Expected cost = ", objective_value(m))

println("\np1:")
for g in G
    println(g, ": ", value(p1[g]))
end

println("\np2 by scenario:")
for s in 1:length(scenarios)
    name = scenarios[s][1]
    println(name)
    for g in G
        println("  ", g, ": ", value(p2[g,s]))
    end
end


Running HiGHS 1.12.0 (git hash: 755a8e027): Copyright (c) 2025 HiGHS under MIT licence terms
LP has 45 rows; 35 cols; 115 nonzeros
Coefficient ranges:
  Matrix  [1e+00, 1e+00]
  Cost    [4e-01, 4e+01]
  Bound   [1e+02, 5e+02]
  RHS     [1e+02, 2e+03]
Presolving model
13 rows, 19 cols, 35 nonzeros  0s
6 rows, 11 cols, 16 nonzeros  0s
4 rows, 5 cols, 8 nonzeros  0s
Presolve reductions: rows 4(-41); columns 5(-30); nonzeros 8(-107) 
Solving the presolved LP

Performed postsolve
Solving the original LP from the solution after postsolve

Model status        : Optimal
Objective value     :  1.2200000000e+04
P-D objective error :  0.0000000000e+00
HiGHS run time      :          0.01
Expected cost = 12200.0

p1:
Biomass: 0.0
Hydro: 315.0
Geo: 0.0
NG_CCGT: 100.0
NG_CT: 100.0
Wind: 135.0
Solar: 450.0

p2 by scenario:
S1
  Biomass: 0.0
  Hydro: 405.0
  Geo: 0.0
  NG_CCGT: 100.0
  NG_CT: 100.0
  Wind: 120.0
  Solar: 475.0
S2
  Biomass: 0.0
  Hydro: 475.0
  Geo: 0.0
  NG_CCGT: 100.0
  NG_CT: 100.0


## References

List any external references consulted, including classmates.

Used CoPilot for debugging